In [4]:
import polars as pl
import urllib3
from elasticsearch import Elasticsearch, helpers
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
import h3
from support_fun import nearby_airports

In [5]:
def get_airports_list(city_id):
    search_params = {
        "id": city_id,
        "start_date_string": "2025-10-15",
        "end_date_string": "2025-10-22",
        "activity_tag": "summer",
    }


    airports_list = nearby_airports(search_params).select(['iata_code','name','iso_country']).to_dicts()

    return airports_list
get_airports_list(1688374696)

In [2]:
pl.read_parquet('raw_files/airports_filtered_h3.parquet')

In [14]:
cities_raw = pl.read_parquet('worldcitiesh3.parquet')

In [15]:
df = cities_raw.select('id','city_ascii','country', 'admin_name')

In [16]:
from elasticsearch import Elasticsearch, helpers
import polars as pl

# Initialize Elasticsearch client
es = Elasticsearch(
    "https://127.0.0.1:9200",  # Replace with your Elasticsearch server URL
    basic_auth=("elastic", "rade123"),
    verify_certs=False  # Set to True if you have a valid SSL certificate
)
# Define index name
INDEX_NAME = "cities"

# Delete index if it exists
if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)

# ✅ Create index with edge_ngram tokenizer
es.indices.create(index=INDEX_NAME, body={
    "settings": {
        "analysis": {
            "tokenizer": {
                "edge_ngram_tokenizer": {
                    "type": "edge_ngram",
                    "min_gram": 2,
                    "max_gram": 10,
                    "token_chars": ["letter"]
                }
            },
            "analyzer": {
                "custom_edge_ngram_analyzer": {
                    "type": "custom",
                    "tokenizer": "edge_ngram_tokenizer",
                    "filter": ["lowercase"]
                }
            }
        }
    },
    "mappings": {
        "properties": {
            "id": {"type": "integer"},
            "city_ascii": {
                "type": "text",
                "analyzer": "custom_edge_ngram_analyzer"
            }
        }
    }
})


actions = [
    {
        "_index": INDEX_NAME,
        "_id": row["id"],
        "_source": row
    }
    for row in df.to_dicts()
]

helpers.bulk(es, actions)
print("Indexing complete.")



In [17]:
def search_city_ids(query: str):
    """
    Efficiently search for city ids using the substring query.
    If query length <= 3, use standard tokenizer.
    If query length >= 4 and no results, use fuzzy tokenizer.
    """
    base_query = {
        "_source": ["id"],  # Only fetch 'id' field
        "query": {
            "match": {
                "city_ascii": {
                    "query": query,
                    "operator": "and"
                }
            }
        },
        "sort": [{"_score": {"order": "desc"}}]  # Sort by relevance score in descending order
    }

    # Check the query length and decide whether to use fuzziness
    if len(query) >= 4:
        base_query["query"]["match"]["city_ascii"]["fuzziness"] = "1"

    response = es.search(index=INDEX_NAME, body=base_query)
    hits = response['hits']['hits']

    # If no results for query >=4, use fuzzy search
    if not hits and len(query) >= 4:
        fuzzy_query = {
            "_source": ["id"],  # Fetch only 'id' field
            "query": {
                "match": {
                    "city_ascii": {
                        "query": query,
                        "fuzziness": "1"
                    }
                }
            },
            "sort": [{"_score": {"order": "desc"}}]  # Sort by score for fuzzy match
        }
        response = es.search(index=INDEX_NAME, body=fuzzy_query)
        hits = response['hits']['hits']

    # Return just the list of ids (sorted by highest score first)
    return [hit["_source"]["id"] for hit in hits]



In [18]:
query = "Bud"

indexed_cities = cities_raw.filter(pl.col('id').is_in(search_city_ids(query)))
json_results = indexed_cities.to_dicts()
results = [{'city':city['city_ascii'],'country':city['country'], 'id':city['id'], 'admin_name':city['admin_name']} for city in json_results[0:10]]

In [12]:
results